<a href="https://colab.research.google.com/github/rudalshan0412-code/attention-is-all-you-need-pytorch/blob/main/02)_Multi_head_Attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결

from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 프로젝트 경로 설정
from pathlib import Path
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/attention_is_all_you_need")
SRC_DIR = PROJECT_ROOT / "src"

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
SRC_DIR.mkdir(parents=True, exist_ok=True)

# src 패키지를 import할 수 있도록 프로젝트 루트를 Python 경로에 추가
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_DIR:", SRC_DIR)

PROJECT_ROOT: /content/drive/MyDrive/attention_is_all_you_need
SRC_DIR: /content/drive/MyDrive/attention_is_all_you_need/src


In [ ]:
# ScaledDotProductAttention 확인

from src.attention import ScaledDotProductAttention

attention = ScaledDotProductAttention()

print(attention)

ScaledDotProductAttention()


In [ ]:
'''
 논문 상으로는 multi-head attention을 수행 하는 경우 여러 head의 single-attention을 병렬로 수행하여 concaterate한 다음, output 가중치를 곱하는 것으로 나타나 있다.
 (head 별로 독립된 linear을 만드는 것)
 그러나, 구현상의 효울성을 위해 큰 linear 한개에 d_model 차원 전체를 한번에 계산한 다음 텐서를 (num_heads, head_dim) 형태로 reshape 및 transpose한다.
 따라서, d_model % num_heads == 0 이여야만 한다.
 '''

'\n 논문 상으로는 multi-head attention을 수행 하는 경우 여러 head의 single-attention을 병렬로 수행하여 concaterate한 다음, output 가중치를 곱하는 것으로 나타나 있다.\n (head 별로 독립된 linear을 만드는 것)\n 그러나, 구현상의 효울성을 위해 큰 linear 한개에 d_model 차원 전체를 한번에 계산한 다음 텐서를 (num_heads, head_dim) 형태로 reshape 및 transpose한다.\n 따라서, d_model % num_heads == 0 이여야만 한다.\n '

In [ ]:
'''
입력 Q의 shape 이 (B, query_len, d_model) 일 때,
Q를 W_Q에 통과시켜도 출력 차원은 d_model임으로 shape는 유지된다.
이후 d_model을 num_heads와 head_dim으로 나눈다.(d_model을 하나의 커다란 벡터로 취급하는 대신 서로 다른 head로 보기 위함)
현재 shape == (B, query_len, num_heads, head_dim)

현재 ScaleDotProductAttention이 기대하는 구조는 마지막 두 차원이 (sequence_length, vector_dimension)(단어 n개와 그에 대한 벡터의 길이 = (query_len, d_v))이기에,
(B, num_heads, query_len, head_dim) 형태로 두는 것이 더 편하다.
따라서, transpose(1, 2)를 수행하여 형태를 맞춰준다.

따라서, 앞쪽 (B, num_heads)는 여러개의 batch로 취급하고, 마지막 두 차원 (query_len, head_dim)에 대해서만 attention 계산을 수행한다.
'''

'\n입력 Q의 shape 이 (B, query_len, d_model) 일 때,\nQ를 W_Q에 통과시켜도 출력 차원은 d_model임으로 shape는 유지된다.\n이후 d_model을 num_heads와 head_dim으로 나눈다.(d_model을 하나의 커다란 벡터로 취급하는 대신 서로 다른 head로 보기 위함)\n현재 shape == (B, query_len, num_heads, head_dim)\n\n현재 ScaleDotProductAttention이 기대하는 구조는 마지막 두 차원이 (sequence_length, vector_dimension)이기에, (B, num_heads, query_len, head_dim) 형태로 두는 것이 더 편하다.\n따라서, transpose(1, 2)를 수행하여 형태를 맞춰준다.\n\n따라서, 앞쪽 (B, num_heads)는 여러개의 batch로 취급하고, 마지막 두 차원 (query_len, head_dim)에 대해서만 attention 계산을 수행한다.\n'

In [ ]:
'''
d_model: 하나의 단어(토큰)이 가지는 벡터의 크기(논문 상으로는 512)
num_heads: head의 개수(논문 상으로는 8)
head_dim: 하나의 head가 가지는 벡터의 크기 (= d_model / num_heads)
W_Q: 입력 데이터를 query 벡터로 변환해주는 가중치 행렬 shape = (d_model, d_model)
W_K: 입력 데이터를 key 벡터로 변환해주는 가중치 행렬 shape = (d_model, d_model)
W_V: 입력 데이터를 value 벡터로 변환해주는 가중치 행렬 shape = (d_model, d_model)
'''

In [ ]:
# 작은 Tensor로 Head 분할 직접 확인

import torch

x = torch.tensor(
    [
        [
            [1, 2, 3, 4],
            [5, 6, 7, 8],
        ]
    ],
    dtype=torch.float32,
)

print("원본 x")
print(x)
print("shape:", x.shape)

# num_heads = 2, head_dim = 2 로 나누기
print("\nnum_heads = 2, head_dim = 2 로 나누기")
batch_size = 1
seq_len = 2
num_heads = 2
head_dim = 2

x_split = x.view(
    batch_size,
    seq_len,
    num_heads,
    head_dim,
)
# .view()는 먼저 한줄로 늘어틀인 다음 뒤쪽의 숫자부터의 그룹으로 묶어준다.

print(x_split)
print("shape:", x_split.shape)

# attention에 넣기 위해 순서 바꾸기
print("\nafter transpose")
x_heads = x_split.transpose(1, 2)

print(x_heads)
print("shape:", x_heads.shape)

# head 별로 확인

print("\nhead 별로 확인")
print("Head 1:")
print(x_heads[0, 0])
print("Head 2:")
print(x_heads[0, 1])

원본 x
tensor([[[1., 2., 3., 4.],
         [5., 6., 7., 8.]]])
shape: torch.Size([1, 2, 4])

num_heads = 2, head_dim = 2 로 나누기
tensor([[[[1., 2.],
          [3., 4.]],

         [[5., 6.],
          [7., 8.]]]])
shape: torch.Size([1, 2, 2, 2])

after transpose
tensor([[[[1., 2.],
          [5., 6.]],

         [[3., 4.],
          [7., 8.]]]])
shape: torch.Size([1, 2, 2, 2])

head 별로 확인
Head 1:
tensor([[1., 2.],
        [5., 6.]])
Head 2:
tensor([[3., 4.],
        [7., 8.]])


In [ ]:
# Multi-Head Attention

import torch
import torch.nn as nn

from src.attention import ScaledDotProductAttention


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__() # nn.Module 상속

        assert d_model % num_heads == 0 # False면 에러 일으킴

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        # .Linear(d_model, d_model)은 d_model 크기의 백터를 받아서 d_model 크기의 가중치와 편향을 가진 벡터를 생성함

        self.attention = ScaledDotProductAttention()

        self.W_O = nn.Linear(d_model, d_model) # Output weigjt

    def split_heads(self, x):
        """
        x:
            (batch_size, seq_len, d_model)

        Returns:
            (batch_size, num_heads, seq_len, head_dim)
        """

        batch_size, seq_len, _ = x.shape

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim,
        )

        x = x.transpose(1, 2)

        return x

    def forward(self, Q, K, V, mask=None): # 이때 QKV는 ()
        """
        Args:
            Q:
                (batch_size, query_len, d_model)

            K:
                (batch_size, key_len, d_model)

            V:
                (batch_size, key_len, d_model)

            mask:
                Attention scores에 broadcast 가능한 mask

        Returns:
            output:
                (batch_size, query_len, d_model)

            attention_weights:
                (batch_size, num_heads, query_len, key_len)
        """

        # 1. Linear Projection
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)

        # 2. Head 분할
        Q = self.split_heads(Q) # 여기서 벡터가 4개인 형태로 바뀜
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 3. 각 Head에서 Scaled Dot-Product Attention
        attention_output, attention_weights = self.attention(
            Q,
            K,
            V,
            mask,
        ) # 가중치를 더한 QKV를 연산

        # attention_output:
        # (batch_size, num_heads, query_len, head_dim)

        # 4. Head들을 다시 하나로 합치기
        attention_output = attention_output.transpose(1, 2) # (batch_size, query_len, num_heads, head_dim)

        # (batch_size, query_len, num_heads, head_dim)

        attention_output = attention_output.contiguous() # 벡터의 위치를 바꾸었을 때 실제 메모리 속 배치 순서가 그대로인 경우가 있어서 이를 정렬해줌

        batch_size, query_len, _, _ = attention_output.shape

        concat_output = attention_output.view(
            batch_size,
            query_len,
            self.d_model,
        )

        # 5. Output Linear Projection
        output = self.W_O(concat_output)

        return output, attention_weights

In [ ]:
# 파일로 저장

%%writefile /content/drive/MyDrive/attention_is_all_you_need/src/multi_head_attention.py

import torch
import torch.nn as nn

from src.attention import ScaledDotProductAttention


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)

        self.attention = ScaledDotProductAttention()

        self.W_O = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        """
        x:
            (batch_size, seq_len, d_model)

        Returns:
            (batch_size, num_heads, seq_len, head_dim)
        """

        batch_size, seq_len, _ = x.shape

        x = x.view(
            batch_size,
            seq_len,
            self.num_heads,
            self.head_dim,
        )

        x = x.transpose(1, 2)

        return x

    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q:
                (batch_size, query_len, d_model)

            K:
                (batch_size, key_len, d_model)

            V:
                (batch_size, key_len, d_model)

            mask:
                Attention scores에 broadcast 가능한 mask

        Returns:
            output:
                (batch_size, query_len, d_model)

            attention_weights:
                (batch_size, num_heads, query_len, key_len)
        """

        # 1. Linear Projection
        Q = self.W_Q(Q)
        K = self.W_K(K)
        V = self.W_V(V)

        # 2. Head 분할
        Q = self.split_heads(Q)
        K = self.split_heads(K)
        V = self.split_heads(V)

        # 3. Scaled Dot-Product Attention
        attention_output, attention_weights = self.attention(
            Q,
            K,
            V,
            mask,
        )

        # 4. Head 병합
        attention_output = attention_output.transpose(1, 2)
        attention_output = attention_output.contiguous()

        batch_size, query_len, _, _ = attention_output.shape

        concat_output = attention_output.view(
            batch_size,
            query_len,
            self.d_model,
        )

        # 5. Output Linear Projection
        output = self.W_O(concat_output)

        return output, attention_weights

Writing /content/drive/MyDrive/attention_is_all_you_need/src/multi_head_attention.py


In [ ]:
# 저장한 class import

from src.multi_head_attention import MultiHeadAttention

mha = MultiHeadAttention(
    d_model=8,
    num_heads=2,
)

print(mha)

print("d_model:", mha.d_model)
print("num_heads:", mha.num_heads)
print("head_dim:", mha.head_dim)

MultiHeadAttention(
  (W_Q): Linear(in_features=8, out_features=8, bias=True)
  (W_K): Linear(in_features=8, out_features=8, bias=True)
  (W_V): Linear(in_features=8, out_features=8, bias=True)
  (attention): ScaledDotProductAttention()
  (W_O): Linear(in_features=8, out_features=8, bias=True)
)
d_model: 8
num_heads: 2
head_dim: 4


In [ ]:
# 기본 동작 테스트

import torch

torch.manual_seed(42)

batch_size = 2
query_len = 3
key_len = 4
d_model = 8
num_heads = 2

Q = torch.randn(batch_size, query_len, d_model)
K = torch.randn(batch_size, key_len, d_model)
V = torch.randn(batch_size, key_len, d_model)

mha = MultiHeadAttention(
    d_model=d_model,
    num_heads=num_heads,
)

output, attention_weights = mha(Q, K, V)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

print()

print("output shape:", output.shape)
print("attention_weights shape:", attention_weights.shape)

Q shape: torch.Size([2, 3, 8])
K shape: torch.Size([2, 4, 8])
V shape: torch.Size([2, 4, 8])

output shape: torch.Size([2, 3, 8])
attention_weights shape: torch.Size([2, 2, 3, 4])


In [ ]:
# 핵심 Shape assert 테스트

assert output.shape == (
    batch_size,
    query_len,
    d_model,
)

assert attention_weights.shape == (
    batch_size,
    num_heads,
    query_len,
    key_len,
)

print("Shape 테스트 통과")

Shape 테스트 통과


In [ ]:
# Attention Weight 합이 1인지 확인(softmax 했으니까)

weight_sums = attention_weights.sum(dim=-1)

print(weight_sums)
print("shape:", weight_sums.shape)

assert torch.allclose(
    weight_sums,
    torch.ones_like(weight_sums),
    atol=1e-6,
)

print("Attention weight 합 테스트 통과")

tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
shape: torch.Size([2, 2, 3])
Attention weight 합 테스트 통과


In [ ]:
# 내부 Shape 변화 한단계씩 추적

torch.manual_seed(42)

batch_size = 2
query_len = 3
key_len = 4
d_model = 8
num_heads = 2

Q = torch.randn(batch_size, query_len, d_model)
K = torch.randn(batch_size, key_len, d_model)
V = torch.randn(batch_size, key_len, d_model)

mha = MultiHeadAttention(
    d_model=d_model,
    num_heads=num_heads,
)

# -----------------------------------
# 1. 입력
# -----------------------------------

print("[입력]")

print("Q:", Q.shape)
print("K:", K.shape)
print("V:", V.shape)


# -----------------------------------
# 2. Linear Projection
# -----------------------------------

Q_projected = mha.W_Q(Q)
K_projected = mha.W_K(K)
V_projected = mha.W_V(V)

print("\n[Linear Projection 후]")

print("Q_projected:", Q_projected.shape)
print("K_projected:", K_projected.shape)
print("V_projected:", V_projected.shape)


# -----------------------------------
# 3. Head 분할
# -----------------------------------

Q_heads = mha.split_heads(Q_projected)
K_heads = mha.split_heads(K_projected)
V_heads = mha.split_heads(V_projected)

print("\n[Head 분할 후]")

print("Q_heads:", Q_heads.shape)
print("K_heads:", K_heads.shape)
print("V_heads:", V_heads.shape)


# -----------------------------------
# 4. Scaled Dot-Product Attention
# -----------------------------------

attention_output, attention_weights = mha.attention(
    Q_heads,
    K_heads,
    V_heads,
)

print("\n[Scaled Dot-Product Attention 후]")

print("attention_output:", attention_output.shape)
print("attention_weights:", attention_weights.shape)


# -----------------------------------
# 5. Head 순서 복원
# -----------------------------------

merged = attention_output.transpose(1, 2)

print("\n[transpose 후]")

print("merged:", merged.shape)


# -----------------------------------
# 6. Head concat
# -----------------------------------

merged = merged.contiguous()

concat_output = merged.view(
    batch_size,
    query_len,
    d_model,
)

print("\n[Head concat 후]")

print("concat_output:", concat_output.shape)


# -----------------------------------
# 7. W_O
# -----------------------------------

final_output = mha.W_O(concat_output)

print("\n[W_O 적용 후]")

print("final_output:", final_output.shape)

[입력]
Q: torch.Size([2, 3, 8])
K: torch.Size([2, 4, 8])
V: torch.Size([2, 4, 8])

[Linear Projection 후]
Q_projected: torch.Size([2, 3, 8])
K_projected: torch.Size([2, 4, 8])
V_projected: torch.Size([2, 4, 8])

[Head 분할 후]
Q_heads: torch.Size([2, 2, 3, 4])
K_heads: torch.Size([2, 2, 4, 4])
V_heads: torch.Size([2, 2, 4, 4])

[Scaled Dot-Product Attention 후]
attention_output: torch.Size([2, 2, 3, 4])
attention_weights: torch.Size([2, 2, 3, 4])

[transpose 후]
merged: torch.Size([2, 3, 2, 4])

[Head concat 후]
concat_output: torch.Size([2, 3, 8])

[W_O 적용 후]
final_output: torch.Size([2, 3, 8])


In [ ]:
# Mask 없이 동작 확인

output, attention_weights = mha(
    Q,
    K,
    V,
    mask=None,
)

print(output.shape)
print(attention_weights.shape)

torch.Size([2, 3, 8])
torch.Size([2, 2, 3, 4])


In [ ]:
# 간단한 Mask 테스트

mask = torch.tensor(
    [
        [[[True, True, True, False]]],
        [[[True, True, True, False]]],
    ]
)

print("mask shape:", mask.shape)

# 실행
masked_output, masked_attention_weights = mha(
    Q,
    K,
    V,
    mask=mask,
)

print("output shape:")
print(masked_output.shape)

print()

print("attention_weights shape:")
print(masked_attention_weights.shape)

mask shape: torch.Size([2, 1, 1, 4])
output shape:
torch.Size([2, 3, 8])

attention_weights shape:
torch.Size([2, 2, 3, 4])


In [ ]:
# 가려진 attention weight 확인

print(masked_attention_weights[..., -1])
# 전부 0이여야만 한다

assert torch.allclose(
    masked_attention_weights[..., -1],
    torch.zeros_like(masked_attention_weights[..., -1]),
    atol=1e-6,
)

print("Mask 테스트 통과")

tensor([[[0., 0., 0.],
         [0., 0., 0.]],

        [[0., 0., 0.],
         [0., 0., 0.]]], grad_fn=<SelectBackward0>)
Mask 테스트 통과


In [ ]:
# Mask를 적용해도 나머지 weight 합이 1인지 확인

masked_weight_sums = masked_attention_weights.sum(dim=-1)

print(masked_weight_sums)

# softmax하기 전 마지막 위치가 -inf로 바뀌이게 [.., ..., ..., 0] 이여야만 하며 전체 합은 1이여야한다
assert torch.allclose(
    masked_weight_sums,
    torch.ones_like(masked_weight_sums),
    atol=1e-6,
)

print("Masked Attention weight 합 테스트 통과")

tensor([[[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]],

        [[1.0000, 1.0000, 1.0000],
         [1.0000, 1.0000, 1.0000]]], grad_fn=<SumBackward1>)
Masked Attention weight 합 테스트 통과


In [ ]:
# 최종 통합 테스트

import torch

from src.multi_head_attention import MultiHeadAttention


torch.manual_seed(42)

batch_size = 2
query_len = 3
key_len = 4
d_model = 8
num_heads = 2

Q = torch.randn(batch_size, query_len, d_model)
K = torch.randn(batch_size, key_len, d_model)
V = torch.randn(batch_size, key_len, d_model)

mha = MultiHeadAttention(
    d_model=d_model,
    num_heads=num_heads,
)


# -----------------------------------
# 1. mask=None
# -----------------------------------

output, attention_weights = mha(
    Q,
    K,
    V,
)

assert output.shape == (
    batch_size,
    query_len,
    d_model,
)

assert attention_weights.shape == (
    batch_size,
    num_heads,
    query_len,
    key_len,
)

assert torch.allclose(
    attention_weights.sum(dim=-1),
    torch.ones_like(attention_weights.sum(dim=-1)),
    atol=1e-6,
)


# -----------------------------------
# 2. Mask Broadcasting
# -----------------------------------

mask = torch.tensor(
    [
        [[[True, True, True, False]]],
        [[[True, True, True, False]]],
    ]
)

masked_output, masked_attention_weights = mha(
    Q,
    K,
    V,
    mask=mask,
)

assert masked_output.shape == (
    batch_size,
    query_len,
    d_model,
)

assert masked_attention_weights.shape == (
    batch_size,
    num_heads,
    query_len,
    key_len,
)

assert torch.allclose(
    masked_attention_weights[..., -1],
    torch.zeros_like(masked_attention_weights[..., -1]),
    atol=1e-6,
)

assert torch.allclose(
    masked_attention_weights.sum(dim=-1),
    torch.ones_like(masked_attention_weights.sum(dim=-1)),
    atol=1e-6,
)


print("Multi-Head Attention 모든 기본 테스트 통과")

Multi-Head Attention 모든 기본 테스트 통과
